In [1]:
import numpy as np
import cv2
import pandas as pd
import os

def draw_landmarks_on_blank(data, frame_number, image_size=(500, 500), scale_factor=1.0, output_path="landmarks_frame.jpg"):
    """
    Draws facial landmarks for a given frame on a blank image and saves it, with landmark indices.

    Parameters:
        data (pd.DataFrame): DataFrame with columns ['frame', 'x', 'y', 'landmark_index'] containing landmarks.
        frame_number (int): The frame number to visualize.
        image_size (tuple): Size of the blank image (height, width).
        scale_factor (float): Scaling factor for the coordinates.
        output_path (str): Path to save the output image.

    Returns:
        None
    """
    # Filter data for the given frame
    frame_data = data[data['frame'] == frame_number]

    if frame_data.empty:
        raise ValueError(f"No data found for frame {frame_number}")

    # Create a blank image
    blank_image = np.zeros((image_size[0], image_size[1], 3), dtype=np.uint8)

    # Scale coordinates to fit within the image
    x_coords = (frame_data['x_scaled'] * scale_factor).astype(int)
    y_coords = (frame_data['y_scaled'] * scale_factor).astype(int)

    # Draw landmarks as circles and add landmark indices as text
    for i, (x, y) in enumerate(zip(x_coords, y_coords)):
        # Draw the landmark point
        cv2.circle(blank_image, (x + int(image_size[0] / 2), y + int(image_size[0] / 2)), radius=3, color=(0, 255, 0), thickness=-1)
        # Draw the landmark index next to the point
        cv2.putText(blank_image, str(frame_data.iloc[i]['landmark_index']), 
                    (x + int(image_size[0] / 2) + 5, y + int(image_size[0] / 2) - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1, cv2.LINE_AA)

    # Save the image
    cv2.imwrite(output_path, blank_image)
    print(f"Landmarks visualization saved to: {output_path}")



In [2]:
def visualize_landmarks(image_path, frame_number, landmark_indices=None):
    image_size = (5000, 5000)  # Adjust image size as needed
    scale_factor = image_size[0]  # Scale coordinates to image size (assuming normalized input)

    # Output file path
    output_path = f"{image_path.split('/')[-1].split('.csv')[0]}_{frame_number}.jpg"
    
    if landmark_indices is None:
        data = pd.read_csv(image_path)
    else:
        data = pd.read_csv(image_path)
        data = data[data['landmark_index'].isin(landmark_indices)]
        
    # Save the blank image with landmarks and indices
    draw_landmarks_on_blank(data, frame_number, image_size, 1000, output_path)


In [4]:
left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
left_eye_indices = left_eye_upper_indices + left_eye_lower_indices
right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]
right_eye_indices = right_eye_upper_indices + right_eye_lower_indices
lips_upper_outer= [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
lips_lower_outer= [146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
lips_upper_inner= [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
lips_lower_inner= [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
middle_line_indices= [10,9,6,4,1,152]
nose_tip_index = 1

In [5]:
# for i in range(1):
visualize_landmarks("Data/face_landmarks/F3/Preprocessed/3_29_10.csv", 4800, lips_upper_outer + lips_lower_outer)

Landmarks visualization saved to: 3_29_10_4800.jpg


In [11]:
import numpy as np
import cv2
import pandas as pd

def draw_landmarks_on_video(data, video_path, output_path, scale_factor=1.0, frame_range=None):
    """
    Draws facial landmarks on specified frames of the video and saves the output video.

    Parameters:
        data (pd.DataFrame): DataFrame with columns ['frame', 'x', 'y', 'landmark_index'] containing landmarks.
        video_path (str): Path to the input video.
        output_path (str): Path to save the output video with landmarks.
        scale_factor (float): Scaling factor for the coordinates.
        frame_range (tuple): Tuple specifying the start and end frame numbers (inclusive).
                             If None, it will process all frames.

    Returns:
        None
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Error opening video file {video_path}")

    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    # Determine the frame range
    if frame_range is None:
        frame_range = (0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1)  # Process all frames
        print(frame_range)
    start_frame, end_frame = frame_range
    
    
    frame_number = 0
    while cap.isOpened():
        frame_number += 1
        ret, frame = cap.read()
        if not ret:
            break

        # Skip frames outside the specified range
        if frame_number < start_frame or frame_number > end_frame:
            continue

        # Filter data for the given frame
        frame_data = data[data['frame'] == frame_number]
        
        if not frame_data.empty:
            # Scale coordinates to actual pixel values
            x_coords = (frame_data['x'] * frame_width * scale_factor).astype(int)
            y_coords = (frame_data['y'] * frame_height * scale_factor).astype(int)

            # Draw landmarks as circles and add landmark indices as text
            for i, (x, y) in enumerate(zip(x_coords, y_coords)):
                # Draw the landmark point
                cv2.circle(frame, (x, y), radius=3, color=(0, 255, 0), thickness=-1)
                # Draw the landmark index next to the point
#                 cv2.putText(frame, str(frame_data.iloc[i]['landmark_index']),
#                             (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1, cv2.LINE_AA)

        # Write the frame with landmarks to the output video
        out.write(frame)

    # Release the video capture and writer
    cap.release()
    out.release()
    print(f"Landmarks visualization saved to: {output_path}")


In [15]:
data

,video,frame,landmark_index,x,y
0,3_29_10,0,0,0.467928,0.542121
1,3_29_10,0,1,0.466597,0.484692
2,3_29_10,0,2,0.467861,0.500195
3,3_29_10,0,3,0.460106,0.427637
4,3_29_10,0,4,0.466477,0.468002
...,...,...,...,...,...
5014147,3_29_10,10713,463,0.481520,0.388386
5014148,3_29_10,10713,464,0.476163,0.390378
5014149,3_29_10,10713,465,0.473351,0.391453
5014150,3_29_10,10713,466,0.517759,0.393944


In [16]:
data = pd.read_csv("Data/face_landmarks/F3/Raw/3_29_10.csv")  # DataFrame containing the landmarks
video_path = "Data/Raw/3_29_10.mp4"  # Path to the input video
output_path = "3_29_10.mp4"  # Path for the output video

# Specify frame range to process frames from 100 to 200
frame_range = (0, 200)

draw_landmarks_on_video(data, video_path, output_path, frame_range=None)


(0, 10713)
Landmarks visualization saved to: 3_29_10.mp4


In [66]:
import numpy as np
import cv2
import pandas as pd

def eye_aspect_ratio(eye_landmarks):
    # Compute distances between the vertical landmarks

    vertical_1 = np.linalg.norm(
        eye_landmarks.iloc[5][['x', 'y']].values - 
        eye_landmarks.iloc[9][['x', 'y']].values
    )
    vertical_2 = np.linalg.norm(
        eye_landmarks.iloc[3][['x', 'y']].values - 
        eye_landmarks.iloc[11][['x', 'y']].values
    )
    # Compute distance between the horizontal landmarks
    horizontal = np.linalg.norm(
        eye_landmarks.iloc[1][['x', 'y']].values - 
        eye_landmarks.iloc[2][['x', 'y']].values
    )

    return (vertical_1 + vertical_2) / (2.0 * horizontal)

def draw_landmarks_on_video(data, video_path, output_path, scale_factor=1.0, frame_range=None):
    """
    Draws facial landmarks on specified frames of the video and saves the output video.

    Parameters:
        data (pd.DataFrame): DataFrame with columns ['frame', 'x', 'y', 'landmark_index'] containing landmarks.
        video_path (str): Path to the input video.
        output_path (str): Path to save the output video with landmarks.
        scale_factor (float): Scaling factor for the coordinates.
        frame_range (tuple): Tuple specifying the start and end frame numbers (inclusive).
                             If None, it will process all frames.

    Returns:
        None
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Error opening video file {video_path}")

    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    # Determine the frame range
    if frame_range is None:
        frame_range = (0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1)  # Process all frames
    start_frame, end_frame = frame_range

    # Left and right eye indices (as provided)
    left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
    left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
    right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
    right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Get the current frame number
        frame_number = int(cap.get(cv2.CAP_PROP_POS_FRAMES))

        # Skip frames outside the specified range
        if frame_number < start_frame or frame_number > end_frame:
            continue

        # Filter data for the given frame
        frame_data = data[data['frame'] == frame_number]
        
        if not frame_data.empty:
            # Scale coordinates to fit within the image
            x_coords = (frame_data['x'] * frame_width).astype(int)
            y_coords = (frame_data['y'] * frame_height).astype(int)

            # Draw landmarks as circles and add landmark indices as text
            for i, (x, y) in enumerate(zip(x_coords, y_coords)):
                # Draw the landmark point
                cv2.circle(frame, (x, y), radius=3, color=(0, 255, 0), thickness=-1)
                # Draw the landmark index next to the point
#                 cv2.putText(frame, str(frame_data.iloc[i]['landmark_index']),
#                             (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1, cv2.LINE_AA)

            # Calculate EAR for both eyes
            left_eye_data = frame_data[frame_data['landmark_index'].isin(left_eye_upper_indices + left_eye_lower_indices)]
            right_eye_data = frame_data[frame_data['landmark_index'].isin(right_eye_upper_indices + right_eye_lower_indices)]

            left_ear = eye_aspect_ratio(left_eye_data)
            right_ear = eye_aspect_ratio(right_eye_data)
            
            print(f"{frame_number}, {left_ear}, {right_ear}")

            # Draw EAR values on top of the video frame with background
            cv2.putText(frame, f"Left EAR: {left_ear:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Left EAR: {left_ear:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 1, cv2.LINE_AA)
            
            cv2.putText(frame, f"Right EAR: {right_ear:.2f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Right EAR: {right_ear:.2f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 1, cv2.LINE_AA)

        # Write the frame with landmarks and EAR values to the output video
        out.write(frame)

    # Release the video capture and writer
    cap.release()
    out.release()
    print(f"Landmarks and EAR visualization saved to: {output_path}")


In [67]:
data = pd.read_csv("Data/face_landmarks/F1/Raw/1_01_0.csv")  # DataFrame containing the landmarks
video_path = "Data/Raw/1_01_0.mov"  # Path to the input video
output_path = "1_01_0.mov"  # Path for the output video

# Specify frame range to process frames from 100 to 200
frame_range = (0, 500)

draw_landmarks_on_video(data, video_path, output_path, frame_range=frame_range)


1, 0.034749103523661236, 0.045697656896394705
2, 0.028181068897661476, 0.03198106887616896
3, 0.1614827571856198, 0.14897042600480756
4, 0.16929875431099808, 0.15604246812144645
5, 0.17321894879480398, 0.1551793144138619
6, 0.18030249996815328, 0.1623212462060832
7, 0.1770528425104449, 0.15820911940337887
8, 0.17910318580484375, 0.15821841615422472
9, 0.17851172824294909, 0.15652075266774992
10, 0.17531330536853668, 0.1543641202397383
11, 0.1790580772679407, 0.1582715627775358
12, 0.18187875490699276, 0.1612518430632866
13, 0.18208136232325473, 0.16058767725897208
14, 0.1752655169741627, 0.15237249147355145
15, 0.18102237273117167, 0.15728346240196867
16, 0.1832476619360493, 0.1586354224793899
17, 0.1805988121810884, 0.15691438286787054
18, 0.18747484566111444, 0.16231155148886073
19, 0.18519264719586162, 0.1600275039918577
20, 0.17633124802392844, 0.15345473098944368
21, 0.17446407324902044, 0.15285620772187797
22, 0.17517590631237806, 0.15317502420326407
23, 0.17015200723413793, 0.14